# NLP Job Matcher — Mac MPS / CUDA / CPU midterm demonstration

This notebook demonstrates downloaded training/testing data, base-model inference,
a two-step LoRA training run, and adapter reload. **No accuracy improvement is claimed.**
Every stage prints its real output; the executed copy retains those outputs.

**Mac:** open with the project's .venv Python kernel. Device "auto" chooses MPS
on Apple Silicon when available. You can explicitly choose mps, cuda, or cpu.
MPS/CPU use float32, batch size 1 and a 1,024-token smoke limit; model stages run
in separate processes to release memory between stages. CUDA is also supported.

**Project location:** run inside the repository, or set PROJECT_ROOT in code
cell 1 to your local checkout. On Colab, upload the complete source project or
clone a revision containing these scripts before running. Local execution does
not establish that Colab/CUDA has been tested.

The IT pilot uses source AI labels and a senior job, so it is only a pipeline
check. The demonstration resume is synthetic. Review the limitations at the end.

## 1. Locate the project and capture subprocess output
Output from every command is streamed here and saved to a log. Re-running uses
separate notebook-run outputs and does not replace the original smoke adapter.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, html, importlib.metadata, json, os, platform, re, subprocess, sys, time, zipfile
from IPython.display import HTML, display

PROJECT_ROOT = None  # Example: "/Users/yourname/path/to/NLP_Job_Matcher"
candidates = ([Path(PROJECT_ROOT).expanduser()] if PROJECT_ROOT else [])
candidates += [Path.cwd(), *Path.cwd().parents, Path("/content/NLP_Job_Matcher")]
ROOT = next((p.resolve() for p in candidates if (p / "scripts/train_lora.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the source repository, not just the downloaded notebook.")
os.chdir(ROOT)
RUN_DIR = ROOT / "outputs/notebook_run"
(RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "6")
sys.path.insert(0, str(ROOT / "src"))
PYTHON = sys.executable
RUN_STARTED = datetime.now(timezone.utc).isoformat()

def run_command(arguments, log_name):
    command = [str(arg) for arg in arguments]
    print("Running:", " ".join(command), flush=True)
    lines = []
    with (RUN_DIR / "logs" / log_name).open("w") as log:
        with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                lines.append(line)
            return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command, output="".join(lines))
    print("Completed successfully (exit code 0).", flush=True)
    return "".join(lines)

def show_table(headers, rows):
    head = "".join(f"<th>{html.escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

print("Project:", ROOT)
print("Python:", PYTHON)
print("Run started (UTC):", RUN_STARTED)
print("Operating system:", platform.system(), platform.machine())

Project: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher
Python: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python
Run started (UTC): 2026-11-06T23:23:40.545164+00:00
Operating system: Darwin arm64


## 2. Check dependencies
Existing matching versions are reused. A fresh environment installs only missing
or mismatched training dependencies. No repeated multi-gigabyte PyTorch reinstall
is needed on this Mac. Use Python 3.11 for the locally verified environment.

In [2]:
expected = {}
for line in (ROOT / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.lstrip().startswith("#"):
        name, version = line.strip().split("==", 1)
        expected[name] = version
mismatches = []
for name, version in expected.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        mismatches.append((name, installed, version))
if mismatches:
    print("Installing pinned versions for:", mismatches)
    run_command([PYTHON, "-m", "pip", "install", "--cache-dir", ROOT / ".cache/pip",
                 "-r", ROOT / "requirements.txt"], "dependencies.log")
else:
    print("All pinned training dependencies already installed; reusing them.")
show_table(["Package", "Installed version"],
           [(name, importlib.metadata.version(name)) for name in expected])
print("Python version:", platform.python_version())
run_command([PYTHON, "-m", "pip", "check"], "pip_check.log")

All pinned training dependencies already installed; reusing them.


Package,Installed version
torch,2.8.0
transformers,4.57.1
peft,0.17.1
accelerate,1.10.1
PyYAML,6.0.3


Python version: 3.11.11
Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python -m pip check


No broken requirements found.


Completed successfully (exit code 0).


'No broken requirements found.\n'

## 3. Select the device — screenshot A (optional)
Set DEVICE_REQUEST to auto, mps, cuda, or cpu. MPS is Apple's GPU backend.
An explicitly requested unavailable GPU raises an error instead of silently
claiming GPU execution. The selected device is passed to both inference and training.

In [3]:
import torch, psutil, yaml
from job_matcher.modeling import select_device

DEVICE_REQUEST = "auto"  # Set "mps" to require Apple GPU; "cpu" for a fallback.
DEVICE = select_device(DEVICE_REQUEST)
torch.set_num_threads(min(6, os.cpu_count() or 1))
show_table(["Setting", "Value"], [
    ("Requested device", DEVICE_REQUEST), ("Selected device", DEVICE),
    ("Apple MPS available", torch.backends.mps.is_available()),
    ("CUDA available", torch.cuda.is_available()),
    ("System RAM (GiB)", round(psutil.virtual_memory().total / 1024**3, 1)),
    ("Model precision", "float16" if DEVICE == "cuda" else "float32"),
    ("Smoke batch size", 1), ("Smoke maximum sequence length", 1024),
])
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Device check passed:", DEVICE)

/Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting,Value
Requested device,auto
Selected device,mps
Apple MPS available,True
CUDA available,False
System RAM (GiB),18.0
Model precision,float32
Smoke batch size,1
Smoke maximum sequence length,1024


Device check passed: mps


## 4. Download or verify the model and datasets — screenshot B
Assets already on disk are reused; missing assets are downloaded. The output
labels which happened. SHA-256 hashes are checked against the original saved
manifest when available. Reuse is not presented as a fresh network download.

In [4]:
project = json.loads((ROOT / "configs/project.json").read_text())
original_manifest = ROOT / "reports/evidence/all_manifest.json"
expected_hashes = {}
if original_manifest.exists():
    expected_hashes = {a["path"]: a["sha256"] for a in json.loads(original_manifest.read_text())["assets"]}
existed_before = {path for path in expected_hashes if (ROOT / path).exists()}
run_command([PYTHON, ROOT / "scripts/download_assets.py"], "download_and_verify.log")
manifest = json.loads((ROOT / "outputs/downloads/all_manifest.json").read_text())
assert not manifest["errors"], manifest["errors"]
asset_rows = []
for asset in manifest["assets"]:
    path = ROOT / asset["path"]
    assert path.stat().st_size == asset["bytes"]
    if asset["path"] in expected_hashes:
        assert asset["sha256"] == expected_hashes[asset["path"]], f"Hash mismatch: {path}"
    asset_rows.append((asset["path"], f'{asset["bytes"] / 1024**2:,.2f}',
                       "reused existing file" if asset["path"] in existed_before else "available after download",
                       "PASS" if asset["path"] in expected_hashes else "hash recorded"))
show_table(["Asset", "MiB", "Availability", "SHA-256"], asset_rows)
(RUN_DIR / "download_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("SUCCESS: model and both source datasets are available; download manifest has no errors.")

Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/scripts/download_assets.py


Ready: models/base/Qwen2.5-1.5B-Instruct/LICENSE (11,343 bytes)


Ready: models/base/Qwen2.5-1.5B-Instruct/README.md (4,917 bytes)


Ready: models/base/Qwen2.5-1.5B-Instruct/generation_config.json (242 bytes)


Ready: models/base/Qwen2.5-1.5B-Instruct/config.json (660 bytes)


Ready: models/base/Qwen2.5-1.5B-Instruct/tokenizer_config.json (7,305 bytes)


... (7 outputs left out)

Extracted LinkedIn archive


Completed successfully (exit code 0).


Asset,MiB,Availability,SHA-256
models/base/Qwen2.5-1.5B-Instruct/LICENSE,0.01,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/README.md,0.00,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/config.json,0.00,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/generation_config.json,0.00,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/merges.txt,1.59,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/model.safetensors,"2,944.44",reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/tokenizer.json,6.71,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/tokenizer_config.json,0.01,reused existing file,PASS
models/base/Qwen2.5-1.5B-Instruct/vocab.json,2.65,reused existing file,PASS
data/raw/job_resume_fit/README.md,0.00,reused existing file,PASS


SUCCESS: model and both source datasets are available; download manifest has no errors.


## 5. Load, inspect, and prepare datasets — screenshot C
The generated pilot is separate from final reviewed data. This step regenerates
machine-produced review queues; keep human annotations in separate files.

In [5]:
run_command([PYTHON, ROOT / "scripts/prepare_data.py"], "prepare_data.log")
data_summary = json.loads((ROOT / "outputs/data/dataset_summary.json").read_text())
(RUN_DIR / "dataset_summary.json").write_text(json.dumps(data_summary, indent=2) + "\n")
show_table(["Dataset / subset", "Actual count"], [
    ("Resume-Job Fit source pairs", data_summary["source_rows"]),
    ("Distinct source job descriptions", data_summary["unique_job_descriptions"]),
    ("Explicit IT pilot pairs", data_summary["pilot_rows"]),
    ("LinkedIn source postings", data_summary["linkedin"]["rows"]),
    ("Unreviewed entry-level tech candidates", data_summary["linkedin"]["unreviewed_entry_tech_candidates"]),
])
print("LIMITATION: IT pilot pairs all use a senior infrastructure-manager job, not an entry-level benchmark.")

Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/scripts/prepare_data.py


{


  "source_rows": 2385,


  "categories": {


    "HR": 110,


    "DESIGNER": 107,


... (59 outputs left out)

}


Completed successfully (exit code 0).


Dataset / subset,Actual count
Resume-Job Fit source pairs,2385
Distinct source job descriptions,23
Explicit IT pilot pairs,120
LinkedIn source postings,123849
Unreviewed entry-level tech candidates,1299


LIMITATION: IT pilot pairs all use a senior infrastructure-manager job, not an entry-level benchmark.


## 6. Show loaded records and separate splits — screenshot C continued
The preview shows schema and source record metadata without printing full source
resumes. Every normalized exact resume belongs to only one pilot split. Jobs are
not disjoint: the same senior IT job is used in all pilot splits.

In [6]:
source_path = ROOT / project["resume_dataset"]["local_dir"] / "job_resume_fit.csv"
with source_path.open(newline="", encoding="utf-8-sig") as handle:
    reader = csv.DictReader(handle)
    print("Loaded CSV columns:", reader.fieldnames)
    preview = []
    for row in reader:
        if row["category"] == "INFORMATION-TECHNOLOGY":
            preview.append((row["ID"], row["category"], row["job_text"].splitlines()[0],
                            len(row["resume_text"]), row["ai_match_score"]))
            if len(preview) == 3:
                break
show_table(["Source ID", "Category", "Job title", "Resume characters", "Source AI score"], preview)
splits, split_rows = {}, []
for name in ("train", "validation", "test"):
    path = ROOT / "data/processed/pilot" / f"{name}.jsonl"
    splits[name] = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    assert len(splits[name]) == data_summary["splits"][name]
    split_rows.append((name, len(splits[name]), len({row["resume_id"] for row in splits[name]}),
                       str(path.relative_to(ROOT))))
groups = {name: {row["resume_id"] for row in rows} for name, rows in splits.items()}
assert not (groups["train"] & groups["validation"] or groups["train"] & groups["test"]
            or groups["validation"] & groups["test"])
show_table(["Split", "Pairs", "Unique resumes", "Saved file"], split_rows)
print("PASS: train, validation, and test files load; no normalized exact resume overlaps.")
print("Seed:", project["seed"], "| exact normalized pair deduplication; near-duplicate review remains.")

Loaded CSV columns: ['ID', 'resume_text', 'job_text', 'category', 'job_required_skills', 'resume_skill_list', 'ai_matched_skills', 'ai_match_score', 'skill_string_match_score', 'fuzzy_match_score']


Source ID,Category,Job title,Resume characters,Source AI score
36856210,INFORMATION-TECHNOLOGY,IT Infrastructure Manager,4888,62.82
21780877,INFORMATION-TECHNOLOGY,IT Infrastructure Manager,6491,78.55
33241454,INFORMATION-TECHNOLOGY,IT Infrastructure Manager,4447,67.82


Split,Pairs,Unique resumes,Saved file
train,96,96,data/processed/pilot/train.jsonl
validation,12,12,data/processed/pilot/validation.jsonl
test,12,12,data/processed/pilot/test.jsonl


PASS: train, validation, and test files load; no normalized exact resume overlaps.
Seed: 42 | exact normalized pair deduplication; near-duplicate review remains.


## 7. Show the LoRA training configuration
This is a short execution check, not a complete fine-tuning experiment. Frozen
model weights remain shared with the base model; rank-8 adapters train on q/v
projections. Prompt and padding labels are masked; assistant targets are supervised.

In [7]:
smoke = yaml.safe_load((ROOT / "configs/smoke.yaml").read_text())
smoke["output_dir"] = "models/adapters/notebook_smoke"
smoke["evidence_dir"] = "outputs/notebook_run/training"
smoke_path = RUN_DIR / "smoke.yaml"
smoke_path.write_text(yaml.safe_dump(smoke, sort_keys=False))
print(yaml.safe_dump(smoke, sort_keys=False))
print("Model:", project["model"]["repo_id"])
print("Pinned model revision:", project["model"]["revision"])
print("Training device:", DEVICE)
print("Training implementation: scripts/train_lora.py (included in the submission package)")

data: data/processed/pilot/train.jsonl
output_dir: models/adapters/notebook_smoke
evidence_dir: outputs/notebook_run/training
smoke: true
max_samples: 4
max_steps: 2
epochs: 1
max_length: 1024
learning_rate: 0.0002
batch_size: 1
gradient_accumulation_steps: 1
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
target_modules:
- q_proj
- v_proj
seed: 42

Model: Qwen/Qwen2.5-1.5B-Instruct
Pinned model revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
Training device: mps
Training implementation: scripts/train_lora.py (included in the submission package)


## 8. Run the base model — screenshot D
The same explicitly synthetic resume/job pair is used before and after adaptation.
Raw model text is preserved. Invalid JSON/schema is recorded as a model-output
limitation; it does not mean that the inference process failed to execute.

In [8]:
example = json.loads((ROOT / "examples/pair.json").read_text())
print("DEMONSTRATION INPUT (synthetic, not an evaluation example):")
print(json.dumps(example, indent=2))
base_path = RUN_DIR / "base_prediction.json"
run_command([PYTHON, ROOT / "scripts/infer.py", "--device", DEVICE, "--output", base_path], "base_inference.log")
base = json.loads(base_path.read_text())
assert base["device"] == DEVICE and base["adapter"] is None and base["generated_tokens"] > 0
print("SUCCESS: frozen base model generated a response.")
print("Strict output schema valid:", base["valid_schema"])

DEMONSTRATION INPUT (synthetic, not an evaluation example):
{
  "example_type": "synthetic_demonstration_not_evaluation",
  "resume": "Recent BS Computer Science graduate, 2026. Skills: Python, SQL, Git, pandas. Built a course project cleaning CSV data and querying a SQLite database. Completed one three-month data analytics internship. No professional Tableau experience.",
  "job": "Entry-level Data Analyst. Requires a bachelor's degree in computer science or a related field, Python, SQL, and Git. Zero to one year of experience. Tableau is preferred. Responsibilities include cleaning data and preparing dashboards. Location and work authorization requirements are unspecified."
}
Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/scripts/infer.py --device mps --output /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/outputs/notebook_run/base_prediction.json


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{


  "raw_output": "```json\n{\n  \"score\": 85,\n  \"matching_skills\": [\"Python\", \"SQL\", \"Git\"],\n  \"missing_skills\": [\"Tableau\"],\n  \"eligibility\": \"Eligible\",\n  \"explanation\": \"The resume matches most of the required skills for the entry-level Data Analyst position, including Python, SQL, and Git. It also indicates some relevant experience with data analysis through a course project and an internship. The lack of Tableau experience is noted but does not significantly impact eligibility.\"\n}\n```",


  "parsed_output": null,


  "valid_schema": false,


... (14 outputs left out)

  "generation_seconds": 11.680541082983837


}


Completed successfully (exit code 0).


SUCCESS: frozen base model generated a response.
Strict output schema valid: False


## 9. Execute two LoRA optimizer steps and save the adapter — screenshot E
Training runs in a new process to avoid retaining the previous model in unified
memory. The compact table after the logs is convenient for the written report.

In [9]:
run_command([PYTHON, ROOT / "scripts/train_lora.py", "--config", smoke_path, "--device", DEVICE], "lora_training.log")
training_path = RUN_DIR / "training/summary.json"
training_summary = json.loads(training_path.read_text())
steps = [row for row in training_summary["log_history"] if "loss" in row]
assert len(steps) == smoke["max_steps"] and training_summary["device"] == DEVICE
adapter_dir = ROOT / smoke["output_dir"]
assert (adapter_dir / "adapter_model.safetensors").stat().st_size > 0
assert (adapter_dir / "adapter_config.json").is_file()
show_table(["Completed step", "Loss", "Gradient norm", "Learning rate"],
           [(row["step"], row["loss"], round(row["grad_norm"], 5), row["learning_rate"]) for row in steps])
show_table(["Training evidence", "Value"], [
    ("Device", training_summary["device"]),
    ("Prepared smoke subset size", training_summary["samples"]),
    ("Completed optimizer steps", len(steps)),
    ("Trainable parameters", f'{training_summary["trainable_parameters"]:,}'),
    ("Total parameters including adapter", f'{training_summary["total_parameters"]:,}'),
    ("Training runtime (seconds)", training_summary["metrics"]["train_runtime"]),
    ("Truncated examples in prepared subset", training_summary["truncated_samples"]),
    ("Saved adapter bytes", (adapter_dir / "adapter_model.safetensors").stat().st_size),
])
print("SUCCESS: LoRA training completed and adapter files were saved.")
print("These losses are execution evidence, not proof of improved matching quality.")

Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/scripts/train_lora.py --config /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/outputs/notebook_run/smoke.yaml --device mps


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


  0%|          | 0/2 [00:00<?, ?it/s]


 50%|█████     | 1/2 [00:04<00:04,  4.79s/it]


{'loss': 1.8364, 'grad_norm': 0.8256629109382629, 'learning_rate': 0.0002, 'epoch': 0.25}


... (78 outputs left out)

Completed successfully (exit code 0).


Completed step,Loss,Gradient norm,Learning rate
1,1.8364,0.82566,0.0002
2,1.9196,0.97256,0.0001


Training evidence,Value
Device,mps
Prepared smoke subset size,4
Completed optimizer steps,2
Trainable parameters,"1,089,536"
Total parameters including adapter,"1,544,803,840"
Training runtime (seconds),7.6796
Truncated examples in prepared subset,4
Saved adapter bytes,4372840


SUCCESS: LoRA training completed and adapter files were saved.
These losses are execution evidence, not proof of improved matching quality.


## 10. Reload the saved adapter in a fresh process — screenshot F
A new base model is loaded from disk, then the new adapter is loaded and used on
the same synthetic pair. This verifies that the saved adapter can be used later.

In [10]:
adapted_path = RUN_DIR / "adapter_prediction.json"
run_command([PYTHON, ROOT / "scripts/infer.py", "--device", DEVICE, "--adapter", adapter_dir,
             "--output", adapted_path], "adapter_reload_inference.log")
adapted = json.loads(adapted_path.read_text())
assert Path(adapted["adapter"]).resolve() == adapter_dir.resolve()
assert adapted["device"] == DEVICE and adapted["generated_tokens"] > 0
assert adapted["pair_file"] == base["pair_file"]
assert adapted["model"]["revision"] == base["model"]["revision"]
training_summary.update({
    "adapter_reload_verified": True,
    "adapter_reload_evidence": str(adapted_path.relative_to(ROOT)),
    "adapter_reload_verified_at": datetime.now(timezone.utc).isoformat(),
})
training_path.write_text(json.dumps(training_summary, indent=2) + "\n")
show_table(["Run", "Device", "Generated tokens", "Strict schema valid"], [
    ("Base", base["device"], base["generated_tokens"], base["valid_schema"]),
    ("Reloaded LoRA adapter", adapted["device"], adapted["generated_tokens"], adapted["valid_schema"]),
])
print("SUCCESS: saved adapter reloaded in a fresh process and generated a response.")
print("No accuracy or speed comparison is inferred from these two example runs.")

Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/scripts/infer.py --device mps --adapter /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/models/adapters/notebook_smoke --output /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/outputs/notebook_run/adapter_prediction.json


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{


  "raw_output": "```json\n{\n  \"score\": 85,\n  \"matching_skills\": [\"Python\", \"SQL\", \"Git\"],\n  \"missing_skills\": [\"Tableau\"],\n  \"eligibility\": \"Eligible\",\n  \"explanation\": \"The resume matches most of the required skills for the position, including Python, SQL, and Git. It also indicates some relevant experience with data analysis through a course project and an internship. The lack of Tableau experience is noted but does not significantly impact eligibility.\"\n}\n```",


  "parsed_output": null,


  "valid_schema": false,


... (15 outputs left out)

}


Completed successfully (exit code 0).


Run,Device,Generated tokens,Strict schema valid
Base,mps,110,False
Reloaded LoRA adapter,mps,106,False


SUCCESS: saved adapter reloaded in a fresh process and generated a response.
No accuracy or speed comparison is inferred from these two example runs.


## 11. Check split separation and training labels
These tests check that normalized exact resumes do not cross pilot splits,
truncation retains both input fields, and prompt/padding tokens are excluded
from training loss while the assistant end token remains supervised.

In [11]:
test_output = run_command([PYTHON, "-m", "unittest", "discover", "-s", ROOT / "tests", "-v"], "tests.log")
match = re.search(r"Ran (\d+) tests?", test_output)
assert match and int(match.group(1)) >= 3 and "\nOK" in test_output
tests_passed = int(match.group(1))
print(f"SUCCESS: {tests_passed} pipeline tests passed.")

Running: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/.venv/bin/python -m unittest discover -s /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/tests -v


test_only_completion_has_loss_and_eos_is_supervised (test_pipeline.PipelineTests.test_only_completion_has_loss_and_eos_is_supervised) ... ok


test_prompt_truncation_retains_both_fields (test_pipeline.PipelineTests.test_prompt_truncation_retains_both_fields) ... ok


test_resume_groups_do_not_cross_splits (test_pipeline.PipelineTests.test_resume_groups_do_not_cross_splits) ... ok


----------------------------------------------------------------------


Ran 3 tests in 0.137s


OK


Completed successfully (exit code 0).


SUCCESS: 3 pipeline tests passed.


## 12. Save the evidence and summarize the milestone
The evidence archive contains source code, configuration, logs, package versions,
and the saved adapter. It excludes the multi-gigabyte base model and raw datasets.
The executed notebook and HTML are saved by the runner after this cell finishes.

**Scope:** the IT pilot's single senior job and source AI labels are unsuitable
for claims about entry-level recommendation accuracy. The split checks cover
normalized exact duplicates, not all near duplicates. Any invalid model JSON
remains visible. Final-quality labels, ranking, and performance evaluation remain
future work. Training is task-specific matching; employment is the application domain.

In [12]:
packages = subprocess.run([PYTHON, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
(RUN_DIR / "packages.txt").write_text(packages)
verification = {
    "run_started_at": RUN_STARTED, "completed_at": datetime.now(timezone.utc).isoformat(),
    "platform": platform.system(), "machine": platform.machine(), "device": DEVICE,
    "source_resume_pairs": data_summary["source_rows"],
    "source_job_postings": data_summary["linkedin"]["rows"], "pilot_splits": data_summary["splits"],
    "base_inference_executed": True, "lora_optimizer_steps": len(steps),
    "adapter_saved": True, "adapter_reload_verified": training_summary["adapter_reload_verified"],
    "tests_passed": tests_passed, "base_valid_schema": base["valid_schema"],
    "adapter_valid_schema": adapted["valid_schema"],
    "performance_evaluation_run": False, "data_scope": "senior-job proxy pilot, not an entry-level benchmark",
}
(RUN_DIR / "milestone_verification.json").write_text(json.dumps(verification, indent=2) + "\n")
evidence_zip = RUN_DIR / "midterm_evidence.zip"
with zipfile.ZipFile(evidence_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in [RUN_DIR, adapter_dir, ROOT / "configs", ROOT / "src/job_matcher", ROOT / "scripts"]:
        for path in folder.rglob("*"):
            if path.is_file() and path != evidence_zip and "__pycache__" not in path.parts:
                archive.write(path, path.relative_to(ROOT))
show_table(["Midterm execution requirement", "Observed status"], [
    ("Training/testing datasets downloaded and loadable", "PASS"),
    ("Separate pilot train/validation/test files", str(data_summary["splits"])),
    ("LLM inference runs", "PASS"), ("LoRA optimizer steps", len(steps)),
    ("Adapter save and fresh-process reload", "PASS"), ("Pipeline tests", f"{tests_passed} passed"),
    ("Strict response schema: base / adapter", f'{base["valid_schema"]} / {adapted["valid_schema"]}'),
    ("Matching accuracy evaluated", "No — not required for this midterm"),
])
print("Evidence archive:", evidence_zip)
print("Run verification:", RUN_DIR / "milestone_verification.json")
print("SUCCESS: all pipeline stages above completed. Preserve the notebook with its actual outputs.")

Midterm execution requirement,Observed status
Training/testing datasets downloaded and loadable,PASS
Separate pilot train/validation/test files,"{'train': 96, 'validation': 12, 'test': 12}"
LLM inference runs,PASS
LoRA optimizer steps,2
Adapter save and fresh-process reload,PASS
Pipeline tests,3 passed
Strict response schema: base / adapter,False / False
Matching accuracy evaluated,No — not required for this midterm


Evidence archive: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/outputs/notebook_run/midterm_evidence.zip
Run verification: /Users/williambranam/Desktop/Fall2026/NLP/NLP_Job_Matcher/outputs/notebook_run/milestone_verification.json
SUCCESS: all pipeline stages above completed. Preserve the notebook with its actual outputs.


## Brief report and screenshot guide
Use sections **4–6** for downloaded/loaded datasets and splits, **8** for base
inference, **9** for LoRA settings/losses and saved adapter, and **10** for reload
and output. Section **3** shows MPS selection; section **12** summarizes execution.

State that this is task-specific resume-job matching. Explain that the smoke run
verifies implementation, not model quality. Disclose the senior-job pilot and
strict-schema status shown above. Keyword overlap is a planned supporting metric:
Python/SQL/Excel matched out of required Python/SQL/Excel/Tableau gives 3/4 = 75%.
This percentage differs from the model's overall qualification score.